# Topics and Stances in the Facebook Hearing, Step by Step

| Item | Description |
|---|---|
| Goal | Follow the running example, the Facebook hearing of April 10, 2018 (`CHRG-115shrg37801`), through T2 using the saved production run: its five hearing topics from the summary, its 507 legislator turns cleaned into 1,696 numbered sentences, its 49 stance records, and the placement of each under the codebook. Every model input is rebuilt and checked against the hash the run logged, so the chain is verified without calling the model. |
| Input | The hearing's turns from `hearings_sst.parquet` of D1.4 and its summary from T1.2; `member_turns.parquet`, `stance_turns.parquet`, `stance_records.parquet`, and `track2_assigned.parquet` from T2.2; `codebook_v1.parquet`, `track1_assigned.parquet`, `hearing_topics_v1.parquet`, and `topic_groups.parquet` from T2.1; the call logs of T2.1.1 and T2.2.3 in the Interim folders, read as evidence. |
| Output | Readable copies beside this notebook, kept in the repository because the proposal links to them: `01 - Inputs/numbered_turns.md` and `prompts.md`; `02 - Outputs/topics.json`, `stance_records.json`, `stance_records.md`, and `codebook.json`; `03 - Logs and Verification/call_log.jsonl` and `verification.json`. |
| Variables | • `turn_idx`: the turn's position among the hearing's parsed records, zero-based<br>• `sentences`: the cited sentence numbers, starting at 1 within the cleaned turn<br>• `sim`: the cosine between a record's topic phrase and its nearest codebook topic; 0.68 is the floor |
| Run | Executed on 2026-10-31. No model is called. The copies it writes equal the August export, apart from `verification.json`, which now records the new file locations and the topic groups. |


## Step 1. Import packages


In [1]:
import hashlib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pysbd


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T2 - Topic Extraction and Stance Analysis/T2.5 - Running Example [Facebook-Cambridge Analytica Scandal Hearing]")
NOTEBOOK_NAME = "T2.5.1 - Example - Topics and Stances in the Facebook Hearing Step by Step"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
PACKAGE = "CHRG-115shrg37801"
SECTION = SCRATCH_FOLDER / "T2 - Topic Extraction and Stance Analysis"
T21_PROCESSED = SECTION / "T2.1 - Track 1 - Topic Extraction based on Hearing Summaries" / "Processed"
T21_INTERIM = SECTION / "T2.1 - Track 1 - Topic Extraction based on Hearing Summaries" / "Interim"
T22_PROCESSED = SECTION / "T2.2 - Track 2 - Legislator Stance Extraction" / "Processed"
T22_INTERIM = SECTION / "T2.2 - Track 2 - Legislator Stance Extraction" / "Interim"
SUMMARIES_FOLDER = SCRATCH_FOLDER / "T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings" / "Processed"
HEARINGS_FOLDER = SCRATCH_FOLDER / "D1 - Data Collection and Processing/D1.4 - SST Sample Construction" / "Processed"
EXAMPLE_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER  # the readable copies live beside this notebook
INPUTS_FOLDER = EXAMPLE_FOLDER / "01 - Inputs"
OUTPUTS_FOLDER = EXAMPLE_FOLDER / "02 - Outputs"
LOGS_FOLDER = EXAMPLE_FOLDER / "03 - Logs and Verification"
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Define the prompts

The Track 1 prompt of T2.1.1 and the Track 2 prompt of T2.2.2, unchanged. The hashes in the logs are hashes of the rendered user prompts, so the wording must be exact.


In [3]:
PROMPT_VERSION = "stance-extraction-0.3-dev"  # stamped on every logged call; the production run kept this recorded name
WINDOW_SENTENCES = 40  # a longer turn is sent in consecutive windows of this size, with turn-global sentence numbers
STANCES = ("support", "concern")

SYSTEM = ("You are an expert reader of United States congressional hearing transcripts. "
          "You report only what the text states.")

USER = """Below is one speaking turn by a member of Congress, split into numbered sentences.

Task: find every sentence in which the member takes a directional position on a policy object, and return one record per position.

A record has three fields:
- "sentences": the sentence number(s) expressing the position (1-3 consecutive numbers);
- "topic": the policy object, as a short and specific neutral noun phrase (the policy, program, regulation, technology, practice, or budget item itself; never directional wording; name the specific object, not the agency or field around it);
- "stance": "support" if the member endorses, advocates, or urges adoption, funding, or expansion of the topic; "concern" if the member expresses worry, criticism, or objection toward the topic, or urges its restriction, delay, or scrutiny.

The stance is toward the topic exactly as you phrase it. Before returning a record, check that it reads sensibly as "the member supports [topic]" or "the member is concerned about [topic]". If the topic is a problem or harm, the stance toward it is "concern", never "support".

The topic must be the object the member takes a position ON. Entities mentioned only as examples, comparisons, or asides are not topics, and a bare number or amount is never a topic. If several consecutive sentences express the same position on the same topic, return one record covering them, not one record per sentence.

Example: "Youth vaping is an epidemic we cannot ignore." -> {{"sentences": [n], "topic": "youth e-cigarette use", "stance": "concern"}}
Example: "This bill funds the research our labs badly need." -> {{"sentences": [n], "topic": "research funding bill", "stance": "support"}}

Do not return records for: procedure (recognitions, time, the record), courtesies, introductions, logistics, questions asked without an expressed position, or factual statements without evaluation. If no sentence takes a directional position, return an empty list. When in doubt, do not extract.

--- TURN (Speaker: {speaker}) ---
{numbered}
--- END OF TURN ---

Return only a JSON list of records. Your very first output character must be [."""

# Directional words that must not appear inside a topic phrase; direction belongs in the stance field.
# The lint flags a phrase for review. It does not rewrite it.
DIRECTIONAL_LINT = re.compile(
    r"\b(?:ban(?:ning)?|block(?:ing)?|stop(?:ping)?|halt(?:ing)?|"
    r"oppos(?:e|ing|ition)|support(?:ing)?|defend(?:ing)?|protect(?:ing|ion)?|"
    r"against|anti|pro|promot(?:e|ing|ion)|expand(?:ing)?|restrict(?:ing|ion)?|"
    r"repeal(?:ing)?|eliminat(?:e|ing|ion))\b",
    re.IGNORECASE,
)


def lint_topic(phrase):
    """The directional word a topic phrase contains, or None if clean."""
    match = DIRECTIONAL_LINT.search(phrase or "")
    return match.group(0) if match else None


TRACK1_PROMPT_VERSION = "summary-topics-0.2-dev"  # versioned apart from the stance prompt; the production run kept this recorded name
TRACK1_MAX_TOPICS = 5

TRACK1_USER = """Below is a summary of one United States congressional hearing.

Task: list the main topics the hearing covered, ordered from most to least central.

A topic is a short and specific neutral noun phrase: the policy, program, regulation, technology, practice, or budget item itself; never directional wording; name the specific object, not the agency or field around it. An agency, company, or country is never a topic by itself; name the policy object involving it. A bill counts as a topic by its subject or name, never as a bare bill number such as "H.R. 1". A bare number or amount is never a topic. Entities mentioned only as examples, comparisons, or asides are not topics.

List at most {max_topics} topics. Include a topic only if the hearing gave it substantial attention — most summaries support 2 to 4 topics, and padding to {max_topics} is worse than stopping early. Every major subject of the hearing should appear.

--- SUMMARY ---
{summary}
--- END OF SUMMARY ---

Return only a JSON list of topic strings, most central first. Your very first output character must be [."""


def render_track1(summary):
    """The topic-listing prompt for one hearing summary."""
    return TRACK1_USER.format(max_topics=TRACK1_MAX_TOPICS, summary=summary)


## Step 4. Define the sentence rules, the windows, and the parsers

As in T2.2.1 and T2.2.2.


In [4]:
PYSBD_VERSION = "0.3.4"  # the sentence numbers every record points at depend on the segmenter, so its version is pinned
assert pysbd.__version__ == PYSBD_VERSION, f"pysbd {pysbd.__version__} found, {PYSBD_VERSION} required"
SEGMENTER = pysbd.Segmenter(language="en", clean=False)
MIN_SENTENCE_CHARS = 15  # shorter fragments stay in the numbering but are counted as suspect evidence

# From the first of these markers on, the text is written material, not speech: the post-adjournment
# appendix rides on the last speaker's turn, and inserted statements sit inside mid-hearing turns.
TRUNCATE_AT = re.compile(r"\[\s*Whereupon|hearing (?:was |is )?adjourned|\[The [^\]]{0,80}follows[:.\]]", re.IGNORECASE)
STAGE_DIRECTION = re.compile(r"\[[^\]]{2,60}\]\.?")  # a sentence that is only "[Laughter.]" or "[Applause.]"


def is_header(sentence):
    """An inline all-caps heading such as "STATEMENT OF MR. DAVID KEATING"."""
    letters = [c for c in sentence if c.isalpha()]
    return len(sentence) >= 20 and len(letters) >= 10 and sum(c.isupper() for c in letters) / len(letters) >= 0.8


def clean_turn_sentences(sentences, surname, honorific):
    """Apply the four cleaning rules to one turn and count what each rule removed."""
    removed = {"truncated": 0, "label_echo": 0, "stage_direction": 0, "header": 0}
    for i, sentence in enumerate(sentences):  # 1. cut at the first adjournment or inserted-document marker
        if TRUNCATE_AT.search(sentence):
            removed["truncated"] = len(sentences) - i
            sentences = sentences[:i]
            break
    if sentences:  # 2. drop the leading speaker-label echo ("Senator Dorgan.")
        first = sentences[0].rstrip(".")
        if len(sentences[0]) <= 40 and ((surname and first.endswith(surname)) or (honorific and first == honorific)):
            removed["label_echo"] = 1
            sentences = sentences[1:]
    kept = []
    for sentence in sentences:  # 3. drop stage directions; 4. drop all-caps headers
        if STAGE_DIRECTION.fullmatch(sentence.strip()):
            removed["stage_direction"] += 1
        elif is_header(sentence):
            removed["header"] += 1
        else:
            kept.append(sentence)
    return kept, removed


def segment_sentences(text):
    """Split one block of text into sentences; line breaks are flattened first because GPO hard-wraps lines."""
    flat = re.sub(r"\s+", " ", text).strip()
    if not flat:
        return []
    return [s.strip() for s in SEGMENTER.segment(flat) if s.strip()]


def segment_turn(text):
    """Split a turn paragraph by paragraph, so a sentence can never cross a paragraph boundary."""
    sentences = []
    for block in re.split(r"\n\s*\n", text or ""):
        sentences.extend(segment_sentences(block))
    return sentences


def format_numbered(sentences, start=1):
    """The sentences as the model sees them: one per line, numbered from `start`."""
    return "\n".join(f"[{i}] {s}" for i, s in enumerate(sentences, start))


def render_user(sentences, speaker, start=1):
    """The extraction prompt for one turn, or one window of a long turn."""
    return USER.format(speaker=speaker, numbered=format_numbered(sentences, start=start))


def windows(sentences, size=WINDOW_SENTENCES):
    """Yield (first sentence number, sentences) for each window of a turn; a short turn is one window."""
    for offset in range(0, max(len(sentences), 1), size):
        yield offset + 1, sentences[offset:offset + size]


def parse_records(output, n_sentences, start=1):
    """Validate one model output into records and a list of named errors.

    Every check that code can make is made here: JSON shape, field presence, stance vocabulary,
    sentence indices within the window (at most 3, consecutive), and the directional lint on the topic.
    A record that fails a structural check is dropped and counted; a lint hit keeps the record with a flag.
    """
    errors = []
    text = output.strip()
    if text.startswith("```"):  # a markdown fence is tolerated but counted
        errors.append("markdown_fence")
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        raw = json.loads(text)
    except json.JSONDecodeError:
        return [], errors + ["json_parse"]
    if not isinstance(raw, list):
        return [], errors + ["not_a_list"]
    records = []
    for item in raw:
        if not isinstance(item, dict):
            errors.append("record_not_object")
            continue
        topic, stance, sent = item.get("topic"), item.get("stance"), item.get("sentences")
        if isinstance(sent, int):
            sent = [sent]
        if not (isinstance(topic, str) and topic.strip()):
            errors.append("missing_topic")
            continue
        if stance not in STANCES:
            errors.append("bad_stance")
            continue
        if not (isinstance(sent, list) and sent and all(isinstance(i, int) for i in sent)):
            errors.append("bad_sentences_field")
            continue
        sent = sorted(set(sent))
        if sent[0] < start or sent[-1] > start + n_sentences - 1:
            errors.append("index_out_of_range")
            continue
        if len(sent) > 3 or sent[-1] - sent[0] != len(sent) - 1:
            errors.append("indices_not_consecutive")
            continue
        records.append({"sentences": sent, "topic": topic.strip(), "stance": stance, "topic_lint": lint_topic(topic)})
    return records, errors


def merge_adjacent_records(records):
    """Merge records with the same topic and stance whose sentences touch or sit within two sentences; returns the list and the merge count."""
    if not records:
        return records, 0
    ordered = sorted(records, key=lambda r: (r["topic"].strip().lower(), r["stance"], r["sentences"][0]))
    merged, n_merged = [], 0
    for record in ordered:
        if merged:
            prev = merged[-1]
            if (prev["topic"].strip().lower() == record["topic"].strip().lower() and prev["stance"] == record["stance"]
                    and record["sentences"][0] - prev["sentences"][-1] <= 2):
                prev["sentences"] = sorted(set(prev["sentences"] + record["sentences"]))
                n_merged += 1
                continue
        merged.append(dict(record))
    merged.sort(key=lambda r: r["sentences"][0])
    return merged, n_merged


def parse_topics(output):
    """Validate one Track 1 output into ranked topics and a list of named errors.

    Case-insensitive duplicates are dropped and counted, the first occurrence winning; topics beyond
    TRACK1_MAX_TOPICS are dropped and counted; the directional lint keeps the topic with a flag.
    """
    errors = []
    text = output.strip()
    if text.startswith("```"):
        errors.append("markdown_fence")
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        raw = json.loads(text)
    except json.JSONDecodeError:
        return [], errors + ["json_parse"]
    if not isinstance(raw, list):
        return [], errors + ["not_a_list"]
    topics, seen = [], set()
    for item in raw:
        if not (isinstance(item, str) and item.strip()):
            errors.append("topic_not_string")
            continue
        phrase = item.strip()
        if phrase.lower() in seen:
            errors.append("duplicate_topic")
            continue
        if len(topics) >= TRACK1_MAX_TOPICS:
            errors.append("too_many_topics")
            continue
        seen.add(phrase.lower())
        topics.append({"rank": len(topics) + 1, "topic": phrase, "topic_lint": lint_topic(phrase)})
    return topics, errors


## Step 5. Load the hearing's turns and its saved results

Rows of this hearing only, from each saved table, plus every logged call whose document id names the hearing.


In [5]:
def rows_of(path, **kw):
    return pd.read_parquet(path, filters=[("packageId", "==", PACKAGE)], **kw)


hearing = pd.read_parquet(HEARINGS_FOLDER / "hearings_sst.parquet", filters=[("packageId", "==", PACKAGE)], columns=["packageId", "title", "turns"]).iloc[0]
raw = list(hearing["turns"])
summary = rows_of(SUMMARIES_FOLDER / "hearing_summaries.parquet", columns=["packageId", "summary"]).iloc[0]["summary"]
turns = rows_of(T22_PROCESSED / "member_turns.parquet").sort_values("turn_idx").reset_index(drop=True)
status = rows_of(T22_PROCESSED / "stance_turns.parquet").set_index("turn_idx")
original = rows_of(T22_PROCESSED / "stance_records.parquet").sort_values(["turn_idx", "record_idx"]).reset_index(drop=True)
records = rows_of(T22_PROCESSED / "track2_assigned.parquet").sort_values(["turn_idx", "record_idx"]).reset_index(drop=True)
t1_saved = rows_of(T21_INTERIM / "track1" / "topics.parquet").iloc[0]
t1_assigned = rows_of(T21_PROCESSED / "track1_assigned.parquet").sort_values("rank").reset_index(drop=True)
primary = rows_of(T21_PROCESSED / "hearing_topics_v1.parquet").iloc[0]
codebook = pd.read_parquet(T21_PROCESSED / "codebook_v1.parquet")
groups = pd.read_parquet(T21_PROCESSED / "topic_groups.parquet").set_index("topic_id")["group"]
labels = codebook.set_index("topic_id")["label"]

plain_columns = [c for c in original.columns if c != "sentences"]
assert len(records) == len(original) and records[plain_columns].equals(original[plain_columns]), "the placed records are not the assembled records"
assert all(list(a) == list(b) for a, b in zip(records["sentences"], original["sentences"]))

log_lines = []
for path in [T21_INTERIM / "track1" / "status" / "extract.jsonl", *sorted((T22_INTERIM / "status").glob("extract_*.jsonl"))]:
    with path.open() as stream:
        for line in stream:
            if PACKAGE in line and json.loads(line)["document_id"].split(":")[0] == PACKAGE:
                log_lines.append(line)
calls = [json.loads(line) for line in log_lines]
by_turn = defaultdict(list)
for call in calls:
    if call["stage"] == "extract":
        by_turn[int(call["document_id"].split(":")[1])].append(call)
print(f"{hearing['title']}")
print(f"{len(raw)} parsed records; {len(turns)} legislator turns; {len(records)} stance records; {len(calls)} logged calls ({sum(c['stage'] == 'track1' for c in calls)} Track 1, {len(calls) - 1} Track 2)")


FACEBOOK, SOCIAL MEDIA PRIVACY, AND THE USE AND ABUSE OF DATA
903 parsed records; 507 legislator turns; 49 stance records; 508 logged calls (1 Track 1, 507 Track 2)


## Step 6. Verify the Track 1 call

The summary is rendered into the Track 1 prompt; its hash must equal the logged one. The logged output is parsed again and must equal the saved topics.


In [6]:
t1_call, = [c for c in calls if c["stage"] == "track1"]
t1_prompt = render_track1(summary)
assert hashlib.sha256(t1_prompt.encode()).hexdigest() == t1_call["input_sha256"], "Track 1 prompt hash differs"
assert t1_call["prompt_version"] == TRACK1_PROMPT_VERSION
parsed_topics, t1_errors = parse_topics(t1_call["output"])
assert parsed_topics == json.loads(t1_saved["topics_json"]) and ";".join(t1_errors) == t1_saved["errors"]
print("logged output:", t1_call["output"])
print("parsed       :", [t["topic"] for t in parsed_topics], "| errors:", t1_errors or "none")


logged output: ["Facebook data privacy", "content moderation", "election integrity", "Honest Ads Act", "GDPR-style controls"]
parsed       : ['Facebook data privacy', 'content moderation', 'election integrity', 'Honest Ads Act', 'GDPR-style controls'] | errors: none


## Step 7. Rebuild the cleaned sentences and verify every Track 2 call

The legislator turns are selected from the parsed records with the rule of T2.2.1 and cleaned with its rules. Each is rendered window by window into the Track 2 prompt and hashed. Each logged output is parsed and merged again and must equal the saved records, errors, and merge counts.


In [7]:
selected = [i for i, r in enumerate(raw) if r.get("section") == "spoken" and (r.get("is_member") or r.get("bioGuideId") == "0000000")]
assert selected == turns["turn_idx"].tolist(), "the selected turns differ"

removal_totals = Counter()
by_record = defaultdict(list)
for r in records.itertuples():
    by_record[r.turn_idx].append({"sentences": list(r.sentences), "topic": r.topic, "stance": r.stance, "topic_lint": r.topic_lint})
n_calls = 0
for row in turns.itertuples():
    idx = row.turn_idx
    cleaned, removed = clean_turn_sentences(segment_turn(raw[idx]["text"]), row.surname, row.honorific)
    assert cleaned == list(row.sentences) and len(cleaned) == row.n_sentences, f"turn {idx}: sentences differ"
    removal_totals.update(removed)
    speaker = f"{row.honorific or ''} {row.surname or ''}".strip() or "Unidentified"
    saved_calls = sorted(by_turn[idx], key=lambda c: c["index"])
    windows_of_turn = list(windows(cleaned))
    assert len(windows_of_turn) == len(saved_calls), f"turn {idx}: {len(windows_of_turn)} windows, {len(saved_calls)} calls"
    extracted, errors = [], []
    for i, ((start, sentences), call) in enumerate(zip(windows_of_turn, saved_calls)):
        prompt = render_user(sentences, speaker, start=start)
        assert call["index"] == i and call["prompt_version"] == PROMPT_VERSION
        assert hashlib.sha256(prompt.encode()).hexdigest() == call["input_sha256"], f"turn {idx} window {i}: prompt hash differs"
        parsed, flags = parse_records(call["output"], len(sentences), start=start)
        extracted.extend(parsed)
        errors.extend(flags)
    merged, n_merged = merge_adjacent_records(extracted)
    assert merged == by_record[idx], f"turn {idx}: records differ"
    assert len(merged) == status.loc[idx, "n_records"] and n_merged == status.loc[idx, "n_merged"] and ";".join(errors) == status.loc[idx, "errors"]
    n_calls += len(saved_calls)
assert n_calls + 1 == len(calls)

print(f"{len(turns)} turns selected and cleaned as saved; sentences removed by rule: {dict(removal_totals)}; {sum(turns['n_sentences']):,} kept")
print(f"{n_calls} Track 2 prompts rebuilt and matched to the log; {len(records)} records parsed and merged as saved")
print(f"parser errors in this hearing: {dict(Counter(e for v in status['errors'] for e in v.split(';') if e))}; merges: {int(status['n_merged'].sum())}")


507 turns selected and cleaned as saved; sentences removed by rule: {'truncated': 13380, 'label_echo': 501, 'stage_direction': 15, 'header': 44}; 1,696 kept
507 Track 2 prompts rebuilt and matched to the log; 49 records parsed and merged as saved
parser errors in this hearing: {'indices_not_consecutive': 8}; merges: 1


## Step 8. Check the topic assignment and the placement

Track 1: each of the five phrases carries the topic of its cluster. The hearing's primary topic is the one with the most phrases; a tie goes to the lower topic number. Track 2: a record is placed when its cosine is at least 0.68, and its label is the codebook's.


In [8]:
counts = Counter(t for t in t1_assigned["topic_id"] if t >= 0)
assert primary["n"] == counts[primary["topic_id"]] == max(counts.values())
assert primary["topic_id"] == min(t for t, n in counts.items() if n == max(counts.values()))
assert ((records["sim"] >= 0.68) == (records["topic_id"] >= 0)).all()
assert (records["topic_label"] == records["topic_id"].map(labels).fillna("other")).all()

print("Track 1 phrases and their topics:")
for r in t1_assigned.itertuples():
    print(f"  rank {r.rank}: {r.phrase:28s} -> topic {r.topic_id:>3} {r.topic_label} [{groups.get(r.topic_id, 'other')}]")
print(f"primary topic: {primary['topic_id']} {primary['primary_label']} ({primary['n']} of {len(t1_assigned)} phrases; topics {sorted(t for t, n in counts.items() if n == primary['n'])} tie)")
placed = records["topic_id"] >= 0
print(f"\nTrack 2: {placed.sum()} of {len(records)} records placed under a codebook topic, {(~placed).sum()} in the other pool")


Track 1 phrases and their topics:
  rank 1: Facebook data privacy        -> topic  68 consumer data privacy legislation [Privacy and digital consumer protection]
  rank 2: content moderation           -> topic  93 Online advertising and consumer choice policies [Privacy and digital consumer protection]
  rank 3: election integrity           -> topic 100 Voting systems standards and integrity [Cybersecurity and system integrity]
  rank 4: Honest Ads Act               -> topic  93 Online advertising and consumer choice policies [Privacy and digital consumer protection]
  rank 5: GDPR-style controls          -> topic  68 consumer data privacy legislation [Privacy and digital consumer protection]
primary topic: 68 consumer data privacy legislation (2 of 5 phrases; topics [68, 93] tie)

Track 2: 41 of 49 records placed under a codebook topic, 8 in the other pool


## Step 9. Read the chain for one turn

Senator Hassan's turn 772: the numbered sentences the model saw, the record it returned, and where the record was placed.


In [9]:
EXAMPLE_TURN = 772
row = turns.set_index("turn_idx").loc[EXAMPLE_TURN]
print(format_numbered(list(row.sentences)))
print()
call, = by_turn[EXAMPLE_TURN]
print("logged output :", call["output"])
for r in records[records["turn_idx"] == EXAMPLE_TURN].itertuples():
    print(f"placed        : sentences {list(r.sentences)}, {r.stance} toward '{r.topic}' -> topic {r.topic_id} {r.topic_label}, cosine {r.sim:.4f}")


[1] OK.
[2] And I understand the point that you are trying to make here, but here is what I am concerned about.
[3] We have heard this point from you over the last decade-plus since you founded Facebook, and I understand that you founded it pretty much as a solo entrepreneur with your roommate, but now, you know, you are sitting here, the head of a bazillion-dollar company.
[4] And we have heard you apologize numerous times and promise to change, but here we are again, right?
[5] So I really firmly believe in free enterprise, but when private companies are unwilling or unable to do what is necessary, public officials have historically in every industry stepped up to protect our constituents and consumers.
[6] You have supported targeted regulations such as the Honest Ads Act, and that is an important step for election integrity.
[7] I am proud to be a cosponsor of that bill.
[8] But we need to address other broader issues as well.
[9] And today, you have said you would be open to some 

## Step 10. Write the readable copies

The same files the August export wrote. The numbered turns link to the parsed records of the T1 running example; the stance records link back to their turns.


In [10]:
for folder in (INPUTS_FOLDER, OUTPUTS_FOLDER, LOGS_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


def plain(value):
    # numpy scalars and arrays as plain Python values, so JSON writes them as the August export did
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return [plain(v) for v in value]
    return value


def plain_records(table):
    return [{k: plain(v) for k, v in row.items()} for row in table.to_dict("records")]


def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


write_json(OUTPUTS_FOLDER / "topics.json", {"packageId": PACKAGE, "extracted": parsed_topics, "assigned": plain_records(t1_assigned), "primary": {k: plain(v) for k, v in primary.items()}})
write_json(OUTPUTS_FOLDER / "stance_records.json", plain_records(records))
write_json(OUTPUTS_FOLDER / "codebook.json", plain_records(codebook))
(LOGS_FOLDER / "call_log.jsonl").write_text("".join(log_lines), encoding="utf-8")

text = ["# Numbered legislator turns\n\nSaved cleaned inputs for `CHRG-115shrg37801`. "
        "Turn IDs are zero-based indices in the [original parsed records](<../../../T1 - Text Summarization/T1.4 - Running Example [Facebook-Cambridge Analytica Scandal Hearing]/01 - Inputs/parsed_turns.json>); "
        "sentence numbers start at one within each cleaned turn. Turns longer than 40 sentences are sent in consecutive windows.\n"]
for row in turns.itertuples():
    idx = row.turn_idx
    text.append(f"\n## Turn {idx}\n\n{row.honorific} {row.surname}; speaker ID `{row.bioGuideId}`; {row.n_sentences} sentences; "
                f"{status.loc[idx, 'n_records']} saved stance records. Parser errors: `{status.loc[idx, 'errors'] or 'none'}`.\n\n"
                f"```text\n{format_numbered(list(row.sentences))}\n```\n")
(INPUTS_FOLDER / "numbered_turns.md").write_text("".join(text), encoding="utf-8")

numbered = {r.turn_idx: list(r.sentences) for r in turns.itertuples()}
text = ["# Recorded stance outputs and supporting sentences\n\n"
        f"All {len(records)} accepted records, including the {int((~placed).sum())} assigned to `other`. "
        "Topic wording, stance, evidence indices, similarity, and category come from the saved production tables; "
        "the cited sentences are retrieved from the saved cleaned inputs. "
        "These records include model and category-assignment errors; they have not been editorially corrected.\n"]
for r in records.itertuples():
    text.append(f"\n## Turn {r.turn_idx}, record {r.record_idx}\n\n{r.honorific} {r.surname}: **{r.stance}** toward **{r.topic}**. "
                f"Category `{r.topic_id}`: {r.topic_label}; cosine similarity {r.sim:.6f}; wording flag: `{r.topic_lint or 'none'}`. "
                f"[Full turn](<../01 - Inputs/numbered_turns.md#turn-{r.turn_idx}>).\n\n```text\n")
    text.extend(f"[{i}] {numbered[r.turn_idx][i - 1]}\n" for i in r.sentences)
    text.append("```\n")
(OUTPUTS_FOLDER / "stance_records.md").write_text("".join(text), encoding="utf-8")

example = turns.set_index("turn_idx").loc[EXAMPLE_TURN]
(INPUTS_FOLDER / "prompts.md").write_text(
    "# Verified example prompts\n\nThe user prompts below reproduce the hashes in "
    "[the saved call log](<../03 - Logs and Verification/call_log.jsonl>). The shared system prompt is the one of T2.1.1 and T2.2.2.\n\n"
    f"## System prompt\n\n```text\n{SYSTEM}\n```\n\n"
    f"## Track 1: hearing summary\n\nVersion `{TRACK1_PROMPT_VERSION}`.\n\n```text\n{t1_prompt}\n```\n\n"
    f"## Track 2: Senator Hassan, turn {EXAMPLE_TURN}\n\nVersion `{PROMPT_VERSION}`.\n\n```text\n{render_user(list(example.sentences), 'Senator Hassan')}\n```\n",
    encoding="utf-8")

panel = Counter((r.topic_id, r.topic_label, r.stance) for r in records.itertuples() if r.topic_id >= 0)
write_json(LOGS_FOLDER / "verification.json", {
    "packageId": PACKAGE, "model": t1_call["model"],
    "selected_turns": len(turns), "cleaned_sentences": int(turns["n_sentences"].sum()), "removed_sentences_by_rule": dict(removal_totals),
    "verified_user_prompt_hashes": len(calls), "track2_calls": n_calls, "stance_records": len(records),
    "turns_with_stances": int(records["turn_idx"].nunique()),
    "parser_errors": dict(Counter(e for v in status["errors"] for e in v.split(";") if e)), "merged_records": int(status["n_merged"].sum()),
    "assigned_stance_records": int(placed.sum()), "codebook_topics": len(codebook), "topic_groups": int(groups.nunique()),
    "codebook_parquet_sha256": hashlib.sha256((T21_PROCESSED / "codebook_v1.parquet").read_bytes()).hexdigest(),
    "hearing_contributions_to_115th_congress": [{"topic_id": int(k[0]), "topic_label": k[1], "stance": k[2], "n": v} for k, v in sorted(panel.items())],
    "cap_match": None,
    "verification_scope": "Reproduced the turn selection, the sentence cleaning, all extraction user prompts, and the parsing and merging; checked the saved Track 1 assignment, the primary-topic rule, the Track 2 labels, and the 0.68 floor. No embedding model or LLM was run.",
})
written = sorted(p.relative_to(EXAMPLE_FOLDER).as_posix() for p in EXAMPLE_FOLDER.rglob("*") if p.is_file() and p.suffix in (".md", ".json", ".jsonl") and p.name != "README.md")
print(f"wrote {len(written)} files:")
for name in written:
    print("  ", name)


wrote 8 files:
   01 - Inputs/numbered_turns.md
   01 - Inputs/prompts.md
   02 - Outputs/codebook.json
   02 - Outputs/stance_records.json
   02 - Outputs/stance_records.md
   02 - Outputs/topics.json
   03 - Logs and Verification/call_log.jsonl
   03 - Logs and Verification/verification.json


## Step 11. Check the example

- 507 turns, 1,696 sentences, 508 prompt hashes verified, 49 records, 41 placed.
- The hearing has no CAP match, so the CAP comparison of T2.3 does not include it.
- Its contribution to the 115th Congress on topic 68 is printed.


In [11]:
assert len(turns) == 507 and int(turns["n_sentences"].sum()) == 1696 and len(calls) == 508 and len(records) == 49 and int(placed.sum()) == 41
cap_match = pd.read_parquet(SECTION / "T2.3 - Validation of Topics" / "Processed" / "cap_match.parquet")
assert PACKAGE not in set(cap_match["packageId"])
topic_68 = records[records["topic_id"] == 68]["stance"].value_counts().to_dict()
print(f"1) {len(turns)} turns, {int(turns['n_sentences'].sum()):,} sentences, {len(calls)} hashes verified, {len(records)} records, {int(placed.sum())} placed")
print(f"2) no CAP match for this hearing")
print(f"3) topic 68 ({labels[68]}): {topic_68} in the 115th Congress")


1) 507 turns, 1,696 sentences, 508 hashes verified, 49 records, 41 placed
2) no CAP match for this hearing
3) topic 68 (consumer data privacy legislation): {'support': 7, 'concern': 6} in the 115th Congress
